
### About this doc 

`-` 논문리뷰 / AutoML 공부 

`-` 학부수준 

`-` 이번 포스팅에서는 아래의 논문을 정리하겠다. 
- Zoph, B., & Le, Q. V. (2016). Neural architecture search with reinforcement learning. arXiv preprint arXiv:1611.01578.


### 지나친 의인화에 대한 불편함  

`-` 인공지능을 이해할때 가장 경계해야할 부분이 **"지나친 의인화"** 라고 생각한다. 컴퓨터는 **학습**을 하는게 아니라 **계산**을 하는 것이다. **훈련**과 **예측**을 하는게 아니고 **계산**과 **계산**을 하는 것이다. 물론 의인화가 직관적인 이해에는 도움이 되지만 지나친 의인화는 학술적이지 않다는 생각이 든다. 

`-` 이런 점에서 이 논문은 제시하는 기술력에 비해서 의인화가 너무 거창하다는 느낌을 지울 수 없다. 무려 **인공지능을 만드는 인공지능이다** 라는 슬로건으로 나온 논문이라 정말 엄청나게 혁신적인 알고리즘을 제시한 줄 알았는데 그렇지 않았다. 상당히 비효율적으로 보였다 심지어 몇몇 부분에서는 저자들이 다른분야를 제대로 이해하고 쓴건 맞는가? 라는 의문도 들었다. (1) 먼저 RNN 방식을 왜 사용해야 하는지에 대한 이유가 석연치 않다. 저자들이 설명을 하지 않아 그 motivation 을 이해하는것이 쉽지 않다. 적어도 내가 이해하는 한도내에서는 RNN을 쓸 필요가 없다. (2) 저자들이 제시한 방법은 결국 강화학습을 활용하여 아키텍처를 설계하는 하이퍼파라미터를 옵티마이징 하는것인데 다른방법에 비하여 얼마나 효율적으로 옵티마이징 하는지 비교하지 않았다. 애초에 굳이 강화학습을 이용해야 했는지 자체도 의문이다. 



### Toy Example 

`-` 논문을 이해하기 쉬운 장난감예제를 구성. 

`-` 나스가 RNN을 사용하므로, RNN아키텍처에 대한 선행지식이 필요함. 



### 예제소개 

`-` 이해를 돕기 위해서 가벼운 예제를 도입하였다. 개인적으로 이 논문의 방법이 문제가 많다고 생각하지만 우선 이 예제를 설명하는 범위내에서는 이러한 문제점을 언급하지 않겠다. 이 챕터에서는 단지 저자들이 주장하는 NAS가 어떻게 동작하는지 그리고 어떻게 구현할 수 있는지 이해하는데에만 초점을 맞추겠다. 중간중간에 "아니 이걸 이렇게 (비효율적으로) 구한다고?" 싶은것도 많고 저자들의 생각에 비판의 여지도 많겠겠만 (틀린거 같아) 이것은 다음챕터에서 몰아서 논의하고 이번에는 그냥 이해에만 초점을 맞추도록 하자. 

`-` 아래와 같은 데이터셋 $(x_i,y_i)$ 이 있다고 가정하자. 

$$y_i=2 x_ i + \epsilon_i$$

여기에서 $\epsilon_i$ 는 에러텀이다. 위와 같은 자료에서 우리가 학습하고 싶은 것은 

$$\sum_{i=1}^{n}\big(y_i-f(x_i)\big)^2$$

을 최소화 하는 어떠한 함수 $f(x)$ 이다. 그리고 당연히 이때 $f(x)=2x$ 이다. 

`-` $f(x)$를 학습하기 위해서 each observation $i$에 대하여 아래와 같은 구조를 가지는 MLP 를 쌓았다고 가정하자. 

$$y_i = g\big(h(x_i u) v\big) $$

여기에서 $u$, $v$는 weight 이고 $g$ 와 $h$ 는 activation function 이다. $g\big(h(x_i u) v\big)=2x $ 가 되는것이 최적이므로 $g$ 와 $h$ 를 모두 linear activation 으로 설정하고 $uv=2 $ 가 되도록 하는 weight 를 설정하면 된다. 이게 최선이다. 

### NAS의 셋팅: 아키텍처의 선택문제를 

`-` 이 상황에서 우리는 activation 의 선택지를 sigmoid 와 linear 로 주고 적절한 activation 을 선택하게끔 만드는 NAS를 구성할 것이다. 즉 우리의 예제에서 NAS가 선택해야 하는 조합의 수는 아래의 4가지이다. 

**(1)** $h(x)=x$, $g(x)=x$ 

**(2)** $h(x)=x$, $g(x)=\frac{e^x}{1+e^x}$ 

**(3)** $h(x)=\frac{e^x}{1+e^x}$, $g(x)=x$ 

**(4)** $h(x)=\frac{e^x}{1+e^x}$, $g(x)=\frac{e^x}{1+e^x}$ 

참고로 최선의 선택은 당연히 **(1)** 이다. 

`-` NAS는 여기에서 2개의 action 을 취하게 된다. 하나는 $h$를 고르는 action 이고 하나는 $g$를 고르는 action 이다. 편의상 이 action을 각각 $A_1$ 과 $A_2$ 라고 하자. $A_1$, $A_2$ 가 대문자인 이유는 확률변수이기 때문이다. 그리고 $A_1, A_2$의 실현값을 각각 $a_1,a_2$ 라고 하자. 선택할 수 있는 action 은 linear activation 을 고르거나 sigmoid activatio 을 고르거나 둘 중하나 이므로 ${\cal A}=\{{\tt linear},{\tt sigmoid} \}$ 이다. 편의상 ${\cal A}=\{0,1\}$ 이라고 정의하자. 